# 07b · Weather: ERA5 single levels from ARCO-ERA5

The same surface weather as `era5_single` in notebook 07 (boundary-layer height, cloud base, CAPE, CIN, K index, heat fluxes, sunshine, cloud cover, temperature, dew point, wind at 10 and 100 m, gusts, pressure, rain), read from **ARCO-ERA5**, Google's public copy of ERA5 on Cloud Storage. No account, no key and no queue: it reads one global field per variable and hour, many at a time.

For each site it keeps its competition days and the day before each, from 5 to 21 h local solar time. Each month is saved to `database/weather/arco/single/` as soon as it is read, so **if Colab disconnects, Run All again** and it carries on where it stopped. Hours after ARCO's last date are left to the CDS (notebook 07).

Notebook 07 keeps running in its own session for pressure levels and ERA5-Land. The last step here builds `era5_single.parquet` from these files.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
THREADS = 32         # global fields read at a time
SOLAR = "5-21"       # local solar hours kept for each site
REPO = "/content/pta_code"   # the code is written here by step 2b

## 2. Mount Drive

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "site_chips.parquet")), "Run notebook 06 first: database/site_chips.parquet is missing"
subprocess.run(["pip", "install", "-q", "gcsfs", "zarr", "netCDF4"], check=True)
print("Database in", DB)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script (otherwise it would
    keep running in the background)."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        return proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")

## 2b. Write the code

In [ ]:
# The code this notebook runs, written to REPO (same files as in the project repo)
CODE = {
    "pta/__init__.py": "\"\"\"Pre-task Analyzer: data pipeline and models for paragliding competition tasks.\"\"\"\n",
    "pta/grid.py": "\"\"\"The global grid shared by every step: chips aligned with ERA5 and 1 km cells inside them.\n\nERA5 values belong to the 0.25\u00b0 box centred on each grid point, so a chip is the box around one\nERA5 point: centre at multiples of 0.25\u00b0, edges at multiples of 0.25\u00b0 plus 0.125\u00b0. Each chip is\nsplit into 25 x 25 cells of 0.01\u00b0 (1.1 km north-south; east-west shrinks with latitude, 0.79 km at\n45\u00b0). Ids are integers that any step can compute from a latitude and longitude, with no lookup:\n\n    chip_id = chip_row * 1440 + chip_col          (1440 chips around the world)\n    cell_id = cell_row * 36000 + cell_col         (36000 cells around the world)\n\nRows count north from 90.125\u00b0 S and columns east from 180.125\u00b0 W.\n\"\"\"\n\nimport numpy as np\n\nCHIP_DEG = 0.25\nCELLS_PER_CHIP = 25\nCELL_DEG = CHIP_DEG / CELLS_PER_CHIP\nLAT0, LON0 = -90.0 - CHIP_DEG / 2, -180.0 - CHIP_DEG / 2\nCHIP_COLS, CELL_COLS = 1440, 1440 * CELLS_PER_CHIP\n\n\ndef cell_rc(lat, lon):\n    \"\"\"(row, col) of the cell holding each point (arrays or scalars).\"\"\"\n    lon = (np.asarray(lon, float) - LON0) % 360.0 + LON0  # wrap to [-180.125, 179.875)\n    row = np.floor((np.asarray(lat, float) - LAT0) / CELL_DEG + 1e-9).astype(np.int64)\n    col = np.floor((lon - LON0) / CELL_DEG + 1e-9).astype(np.int64)\n    return row, col\n\n\ndef cell_id(lat, lon):\n    row, col = cell_rc(lat, lon)\n    return row * CELL_COLS + col\n\n\ndef chip_id(lat, lon):\n    row, col = cell_rc(lat, lon)\n    return (row // CELLS_PER_CHIP) * CHIP_COLS + col // CELLS_PER_CHIP\n\n\ndef cell_centre(cell):\n    cell = np.asarray(cell, np.int64)\n    row, col = cell // CELL_COLS, cell % CELL_COLS\n    return LAT0 + (row + 0.5) * CELL_DEG, LON0 + (col + 0.5) * CELL_DEG\n\n\ndef chip_of_cell(cell):\n    cell = np.asarray(cell, np.int64)\n    row, col = cell // CELL_COLS, cell % CELL_COLS\n    return (row // CELLS_PER_CHIP) * CHIP_COLS + col // CELLS_PER_CHIP\n\n\ndef chip_bounds(chip):\n    \"\"\"(south, north, west, east) edges of a chip.\"\"\"\n    row, col = divmod(int(chip), CHIP_COLS)\n    south, west = LAT0 + row * CHIP_DEG, LON0 + col * CHIP_DEG\n    return south, south + CHIP_DEG, west, west + CHIP_DEG\n\n\ndef chip_centre(chip):\n    \"\"\"The ERA5 grid point of a chip (latitude, longitude); chip can be an array.\"\"\"\n    chip = np.asarray(chip, np.int64)\n    row, col = chip // CHIP_COLS, chip % CHIP_COLS\n    lat, lon = LAT0 + (row + 0.5) * CHIP_DEG, LON0 + (col + 0.5) * CHIP_DEG\n    return (float(lat), float(lon)) if lat.ndim == 0 else (lat, lon)\n\n\ndef chips_in_box(south, north, west, east):\n    \"\"\"Every chip that overlaps a latitude/longitude box.\"\"\"\n    r0, c0 = cell_rc(south, west)\n    r1, c1 = cell_rc(north, east)\n    rows = range(int(r0) // CELLS_PER_CHIP, int(r1) // CELLS_PER_CHIP + 1)\n    cols = range(int(c0) // CELLS_PER_CHIP, int(c1) // CELLS_PER_CHIP + 1)\n    return [r * CHIP_COLS + c for r in rows for c in cols]\n",
    "scripts/era5.py": "#!/usr/bin/env python3\n\"\"\"Hourly ERA5 and ERA5-Land weather for every site, on the days of its competitions.\n\nAsks the Copernicus Climate Data Store (CDS) for the hours of each competition day and the day\nbefore it (morning heating starts the evening before for soil and boundary layer; ERA5-Land\naccumulations need the previous hour). Sites flown in the same month are grouped into regional\nboxes of up to BOX_DEG degrees, so one request serves several sites. Three datasets:\n\n    single     ERA5 single levels (0.25\u00b0): boundary-layer height, cloud base, CAPE, CIN, heat\n               fluxes, sunshine, cloud cover, temperature, dew point, wind at 10 and 100 m,\n               gusts, pressure, rain\n    levels     ERA5 pressure levels (0.25\u00b0): wind, temperature, humidity and height at 925, 850,\n               700, 600 and 500 hPa\n    land       ERA5-Land (0.1\u00b0): sensible heat flux, skin temperature, soil moisture, albedo,\n               snow cover, sunshine\n\nSingle levels can instead be read from ARCO-ERA5 on Google Cloud Storage (era5_arco.py, much faster\nthan the CDS queue); run this script with --datasets levels,land and the conversion merges both.\n\nRaw files go to WEATHER/raw/<dataset>/, so a second run only asks for what is missing. They are\nthen converted to tidy tables:\n\n    WEATHER/era5_single.parquet   one row per chip (ERA5 point) and UTC hour\n    WEATHER/era5_levels.parquet   the same, levels as columns (u_850, t_700, ...)\n    WEATHER/era5_land.parquet     one row per 0.1\u00b0 ERA5-Land point and UTC hour\n\nFluxes are converted to W/m\u00b2 (positive upward for the heat fluxes, the way pilots think of them:\nthe ground heating the air), temperatures to \u00b0C, pressure to hPa, rain to mm/h, geopotential to\nmetres. The CDS key is read from the CDSAPI_KEY environment variable.\n\n    CDSAPI_KEY=... python3 scripts/era5.py /content/drive/MyDrive/IGCS/database --parallel 6\n\"\"\"\n\nimport argparse\nimport os\nimport sys\nimport time\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nfrom pta import grid  # noqa: E402\n\nCDS_URL = \"https://cds.climate.copernicus.eu/api\"\nBOX_DEG = 25.0           # sites flown in one month are grouped into boxes up to this size\nHOURS = [f\"{h:02d}:00\" for h in range(24)]\nLEVELS = [\"925\", \"850\", \"700\", \"600\", \"500\"]\n\nDATASETS = {\n    \"single\": {\n        \"name\": \"reanalysis-era5-single-levels\",\n        \"variables\": [\n            \"boundary_layer_height\", \"cloud_base_height\", \"convective_available_potential_energy\",\n            \"convective_inhibition\", \"k_index\", \"surface_sensible_heat_flux\", \"surface_latent_heat_flux\",\n            \"surface_solar_radiation_downwards\", \"total_cloud_cover\", \"low_cloud_cover\",\n            \"medium_cloud_cover\", \"high_cloud_cover\", \"2m_temperature\", \"2m_dewpoint_temperature\",\n            \"10m_u_component_of_wind\", \"10m_v_component_of_wind\", \"100m_u_component_of_wind\",\n            \"100m_v_component_of_wind\", \"instantaneous_10m_wind_gust\", \"surface_pressure\",\n            \"total_precipitation\",\n        ],\n        \"extra\": {\"product_type\": [\"reanalysis\"]},\n        \"step\": 0.25,\n    },\n    \"levels\": {\n        \"name\": \"reanalysis-era5-pressure-levels\",\n        \"variables\": [\"u_component_of_wind\", \"v_component_of_wind\", \"temperature\", \"relative_humidity\",\n                      \"geopotential\"],\n        \"extra\": {\"product_type\": [\"reanalysis\"], \"pressure_level\": LEVELS},\n        \"step\": 0.25,\n    },\n    \"land\": {\n        \"name\": \"reanalysis-era5-land\",\n        \"variables\": [\"surface_sensible_heat_flux\", \"skin_temperature\", \"volumetric_soil_water_layer_1\",\n                      \"forecast_albedo\", \"snow_cover\", \"surface_solar_radiation_downwards\"],\n        \"extra\": {},\n        \"step\": 0.1,\n    },\n}\n\n# netCDF short name -> (column, conversion)\nACCUM_HOURLY = 1 / 3600.0     # J/m\u00b2 over one hour -> W/m\u00b2\nCONVERT = {\n    \"blh\": (\"blh_m\", None), \"cbh\": (\"cloud_base_m\", None), \"cape\": (\"cape\", None), \"cin\": (\"cin\", None),\n    \"kx\": (\"k_index\", None),\n    \"sshf\": (\"sensible_heat_wm2\", -ACCUM_HOURLY), \"slhf\": (\"latent_heat_wm2\", -ACCUM_HOURLY),\n    \"ssrd\": (\"sunshine_wm2\", ACCUM_HOURLY),\n    \"tcc\": (\"cloud_total\", None), \"lcc\": (\"cloud_low\", None), \"mcc\": (\"cloud_mid\", None), \"hcc\": (\"cloud_high\", None),\n    \"t2m\": (\"t2m_c\", \"K\"), \"d2m\": (\"dewpoint_c\", \"K\"), \"skt\": (\"skin_temp_c\", \"K\"),\n    \"u10\": (\"u10\", None), \"v10\": (\"v10\", None), \"u100\": (\"u100\", None), \"v100\": (\"v100\", None),\n    \"i10fg\": (\"gust10\", None), \"sp\": (\"pressure_hpa\", 0.01), \"tp\": (\"rain_mmh\", 1000.0),\n    \"swvl1\": (\"soil_moisture\", None), \"fal\": (\"albedo\", None), \"snowc\": (\"snow_cover\", 0.01),\n    \"u\": (\"u\", None), \"v\": (\"v\", None), \"t\": (\"t\", \"K\"), \"r\": (\"rh\", None), \"z\": (\"z\", 1 / 9.80665),\n}\nLAND_ACCUM = {\"sshf\", \"ssrd\"}   # ERA5-Land accumulates from 00 UTC through the day\n\n\n# \u2500\u2500 Plan \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef site_days(db):\n    \"\"\"{site_id: set of dates} (competition days and the day before each).\"\"\"\n    tasks = pd.read_parquet(db / \"tasks.parquet\", columns=[\"task_key\", \"event_key\", \"date\"])\n    links = pd.read_parquet(db / \"event_sites.parquet\")\n    t = tasks.merge(links, on=\"event_key\").dropna(subset=[\"date\"])\n    t[\"date\"] = pd.to_datetime(t[\"date\"], errors=\"coerce\")\n    t = t.dropna(subset=[\"date\"])\n    days = {}\n    for site, g in t.groupby(\"site_id\"):\n        d = set(g[\"date\"].dt.normalize())\n        days[site] = d | {x - pd.Timedelta(days=1) for x in d}\n    return days\n\n\ndef site_boxes(db):\n    \"\"\"{site_id: (south, north, west, east)} of the ERA5 points of the site's chips.\"\"\"\n    chips = pd.read_parquet(db / \"site_chips.parquet\")\n    return {s: (g[\"era5_lat\"].min(), g[\"era5_lat\"].max(), g[\"era5_lon\"].min(), g[\"era5_lon\"].max())\n            for s, g in chips.groupby(\"site_id\")}\n\n\ndef plan(db, box_deg=BOX_DEG, datasets=tuple(DATASETS)):\n    \"\"\"Requests: one per (dataset, month, regional group of sites). The CDS works through a few\n    requests per user at a time, so fewer, larger requests finish sooner. Ordered by dataset\n    (most useful first), then month.\"\"\"\n    days, boxes = site_days(db), site_boxes(db)\n    by_month = {}\n    for site, ds in days.items():\n        if site not in boxes:\n            continue\n        for d in ds:\n            by_month.setdefault((d.year, d.month), {}).setdefault(site, set()).add(d.day)\n    jobs = []\n    for (year, month), sites in sorted(by_month.items()):\n        groups = []  # [box, {site: days}]\n        for site in sorted(sites, key=lambda s: boxes[s][2]):\n            b = boxes[site]\n            for g in groups:\n                u = (min(g[0][0], b[0]), max(g[0][1], b[1]), min(g[0][2], b[2]), max(g[0][3], b[3]))\n                if u[1] - u[0] <= box_deg and u[3] - u[2] <= box_deg:\n                    g[0] = u\n                    g[1][site] = sites[site]\n                    break\n            else:\n                groups.append([b, {site: sites[site]}])\n        for i, (box, members) in enumerate(groups):\n            all_days = sorted(set().union(*members.values()))\n            for key in datasets:\n                jobs.append({\"dataset\": key, \"year\": year, \"month\": month, \"days\": all_days, \"box\": box,\n                             \"sites\": sorted(members), \"name\": f\"{year}-{month:02d}_b{int(box_deg)}_{i:02d}\"})\n    order = {k: i for i, k in enumerate(DATASETS)}\n    return sorted(jobs, key=lambda j: (order[j[\"dataset\"]], j[\"year\"], j[\"month\"], j[\"name\"]))\n\n\ndef request(job):\n    d = DATASETS[job[\"dataset\"]]\n    s, n, w, e = job[\"box\"]\n    pad = d[\"step\"] if job[\"dataset\"] == \"land\" else 0.0   # ERA5-Land points around the chips' edges\n    return {\n        **d[\"extra\"], \"variable\": d[\"variables\"], \"year\": [str(job[\"year\"])], \"month\": [f\"{job['month']:02d}\"],\n        \"day\": [f\"{x:02d}\" for x in job[\"days\"]], \"time\": HOURS,\n        \"area\": [round(n + pad + 0.125 * (pad > 0), 2), round(w - pad - 0.125 * (pad > 0), 2),\n                 round(s - pad - 0.125 * (pad > 0), 2), round(e + pad + 0.125 * (pad > 0), 2)],\n        \"data_format\": \"netcdf\", \"download_format\": \"unarchived\",\n    }\n\n\n# \u2500\u2500 Download \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef raw_path(weather, job):\n    return weather / \"raw\" / job[\"dataset\"] / f\"{job['name']}.nc\"\n\n\ndef have(weather, job):\n    \"\"\"Already downloaded: this request's file, or files from an earlier run with another box size\n    whose boxes, together, cover all of this request's sites for the month (recorded in the\n    .sites.json written next to each file).\"\"\"\n    if raw_path(weather, job).exists():\n        return True\n    import json\n\n    folder = weather / \"raw\" / job[\"dataset\"]\n    covered = set()\n    for meta in folder.glob(f\"{job['year']}-{job['month']:02d}_*.sites.json\"):\n        if meta.with_name(meta.name.replace(\".sites.json\", \".nc\")).exists():\n            covered |= set(json.loads(meta.read_text()))\n    return set(job[\"sites\"]) <= covered\n\n\ndef note_old_files(db, weather):\n    \"\"\"Files from the first runs (12\u00b0 boxes, named <year>-<month>_<n>.nc) get their .sites.json,\n    so a run with other boxes does not ask for them again.\"\"\"\n    import json\n    import re\n\n    old = [p for p in (weather / \"raw\").glob(\"*/*.nc\") if re.fullmatch(r\"\\d{4}-\\d{2}_\\d{2}\\.nc\", p.name)\n           and not p.with_name(p.name.replace(\".nc\", \".sites.json\")).exists()]\n    if not old:\n        return\n    sites = {f\"{j['dataset']}/{j['name'].replace('_b12_', '_')}\": j[\"sites\"] for j in plan(db, 12.0)}\n    for p in old:\n        key = f\"{p.parent.name}/{p.stem}\"\n        if key in sites:\n            p.with_name(p.name.replace(\".nc\", \".sites.json\")).write_text(json.dumps(sites[key]))\n\n\ndef download(job, weather, key):\n    import cdsapi\n\n    dest = raw_path(weather, job)\n    if dest.exists():\n        return job, \"cached\"\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    tmp = dest.with_suffix(\".part\")\n    client = cdsapi.Client(url=CDS_URL, key=key, quiet=True, progress=False)\n    client.retrieve(DATASETS[job[\"dataset\"]][\"name\"], request(job), str(tmp))\n    import json\n    dest.with_name(dest.name.replace(\".nc\", \".sites.json\")).write_text(json.dumps(job[\"sites\"]))\n    tmp.rename(dest)  # may hold a zip of several netCDF files; read() handles both\n    return job, \"ok\"\n\n\n# \u2500\u2500 Convert \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef _datasets(path):\n    \"\"\"The netCDF datasets in a downloaded file (the CDS sends a zip when variables come from\n    several streams, even when asked for unarchived files).\"\"\"\n    import tempfile\n\n    import xarray as xr\n\n    if not zipfile.is_zipfile(path):\n        return [xr.load_dataset(path)]\n    out = []\n    with zipfile.ZipFile(path) as z, tempfile.TemporaryDirectory() as tmp:\n        for m in z.namelist():\n            if m.endswith(\".nc\"):\n                out.append(xr.load_dataset(z.extract(m, tmp)))\n    return out\n\n\ndef _tidy(ds, dataset):\n    \"\"\"One row per point and hour, columns converted (see CONVERT).\"\"\"\n    ds = ds.rename({k: v for k, v in {\"valid_time\": \"time\", \"pressure_level\": \"level\"}.items() if k in ds.dims or k in ds})\n    ds = ds.drop_vars([v for v in (\"number\", \"expver\") if v in ds.variables and v not in ds.dims], errors=\"ignore\")\n    df = ds.to_dataframe().reset_index()\n    keep = [\"time\", \"latitude\", \"longitude\"] + ([\"level\"] if \"level\" in df else [])\n    out = df[keep].copy()\n    for short in ds.data_vars:\n        if short not in CONVERT:\n            continue\n        col, conv = CONVERT[short]\n        x = df[short].astype(\"float64\")\n        if conv == \"K\":\n            x = x - 273.15\n        elif isinstance(conv, float) and not (dataset == \"land\" and short in LAND_ACCUM):\n            x = x * conv\n        out[col] = x\n    if \"level\" in out:\n        out[\"level\"] = out[\"level\"].astype(float).astype(int)\n        values = [c for c in out.columns if c not in (\"time\", \"latitude\", \"longitude\", \"level\")]\n        out = out.pivot_table(index=[\"time\", \"latitude\", \"longitude\"], columns=\"level\", values=values)\n        out.columns = [f\"{c}_{lvl}\" for c, lvl in out.columns]\n        out = out.reset_index()\n    return out\n\n\ndef _deaccumulate_land(df):\n    \"\"\"ERA5-Land fluxes accumulate from 00 UTC: hourly value = this hour's total minus the\n    previous hour's, except at 01 UTC (the first hour of the day's accumulation).\"\"\"\n    df = df.sort_values([\"latitude\", \"longitude\", \"time\"])\n    for col, sign in ((\"sensible_heat_wm2\", -1.0), (\"sunshine_wm2\", 1.0)):\n        if col not in df:\n            continue\n        prev = df.groupby([\"latitude\", \"longitude\"])[col].shift(1)\n        gap = df.groupby([\"latitude\", \"longitude\"])[\"time\"].diff() != pd.Timedelta(hours=1)\n        hourly = np.where(df[\"time\"].dt.hour == 1, df[col], df[col] - prev)\n        hourly = np.where(gap & (df[\"time\"].dt.hour != 1), np.nan, hourly)\n        df[col] = sign * hourly / 3600.0\n    return df\n\n\ndef convert(weather, wanted_chips):\n    \"\"\"Tidy tables from every raw file.\"\"\"\n    for dataset in DATASETS:\n        files = sorted((weather / \"raw\" / dataset).glob(\"*.nc\"))\n        arco = sorted((weather / \"arco\" / dataset).glob(\"*.parquet\"))  # from era5_arco.py (single levels)\n        if not files and not arco:\n            continue\n        frames = []\n        for path in files:\n            parts = [_tidy(ds, dataset) for ds in _datasets(path)]\n            df = parts[0]\n            for p in parts[1:]:\n                df = df.merge(p, on=[\"time\", \"latitude\", \"longitude\"], how=\"outer\")\n            frames.append(df)\n        df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=[\"time\", \"latitude\", \"longitude\"])\n        df = df.drop_duplicates([\"time\", \"latitude\", \"longitude\"], keep=\"last\")\n        if dataset == \"land\":\n            df = _deaccumulate_land(df)\n            df = df.rename(columns={\"latitude\": \"lat\", \"longitude\": \"lon\"})\n            df[\"chip_id\"] = grid.chip_id(df[\"lat\"].to_numpy(), df[\"lon\"].to_numpy())\n            df = df[df[\"chip_id\"].isin(wanted_chips)]\n        else:\n            df[\"chip_id\"] = grid.chip_id(df[\"latitude\"].round(2).to_numpy(), df[\"longitude\"].round(2).to_numpy())\n            df = df[df[\"chip_id\"].isin(wanted_chips)].drop(columns=[\"latitude\", \"longitude\"])\n            if arco:  # the same variables read from ARCO-ERA5; together they cover every hour\n                df = pd.concat(([df] if frames else []) + [pd.read_parquet(p) for p in arco], ignore_index=True)\n                df = df[df[\"chip_id\"].isin(wanted_chips)].drop_duplicates([\"chip_id\", \"time\"], keep=\"last\")\n        cols = [\"chip_id\", \"time\"] + [c for c in df.columns if c not in (\"chip_id\", \"time\")]\n        df = df[cols].sort_values([\"chip_id\", \"time\"]).reset_index(drop=True)\n        for c in df.columns[2:]:\n            if df[c].dtype == \"float64\":\n                df[c] = df[c].astype(\"float32\")\n        df.to_parquet(weather / f\"era5_{dataset}.parquet\", index=False)\n        print(f\"  era5_{dataset}.parquet: {len(df):,} rows, {df['chip_id'].nunique():,} chips, \"\n              f\"{df['time'].dt.normalize().nunique():,} days\", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument(\"db\", type=Path, help=\"database folder (with sites from build_sites.py)\")\n    parser.add_argument(\"--weather\", type=Path, help=\"output folder (default DB/weather)\")\n    parser.add_argument(\"--parallel\", type=int, default=4, help=\"requests queued at the CDS at a time (default 4; \"\n                        \"the CDS works through one or a few per user at a time, so more only wait in its queue)\")\n    parser.add_argument(\"--box\", type=float, default=BOX_DEG, help=f\"largest regional box, degrees (default {BOX_DEG:g})\")\n    parser.add_argument(\"--datasets\", default=\",\".join(DATASETS), help=\"which datasets, in this order (default single,levels,land)\")\n    parser.add_argument(\"--plan-only\", action=\"store_true\", help=\"print the requests and stop\")\n    parser.add_argument(\"--convert-only\", action=\"store_true\", help=\"only rebuild the tables from the raw files\")\n    args = parser.parse_args()\n    weather = args.weather or args.db / \"weather\"\n    weather.mkdir(parents=True, exist_ok=True)\n\n    datasets = tuple(d for d in args.datasets.split(\",\") if d in DATASETS)\n    note_old_files(args.db, weather)\n    jobs = plan(args.db, args.box, datasets)\n    todo = [j for j in jobs if not have(weather, j)]\n    months = len({(j[\"year\"], j[\"month\"]) for j in jobs})\n    print(f\"ERA5: {len(jobs) // len(datasets):,} regional requests in {months} months \u00d7 {len(datasets)} datasets; \"\n          f\"{len(jobs) - len(todo):,} files already here, {len(todo):,} to ask for\", flush=True)\n    if args.plan_only:\n        for b in (12, 20, 25, 35):\n            print(f\"  boxes of {b}\u00b0: {len(plan(args.db, b, datasets[:1])):,} requests per dataset\")\n        for j in [j for j in jobs if j[\"dataset\"] == datasets[0]][:10]:\n            print(f\"  {j['name']}: {len(j['sites'])} sites, {len(j['days'])} days, box {tuple(round(x, 2) for x in j['box'])}\")\n        return\n\n    if todo and not args.convert_only:\n        key = os.environ.get(\"CDSAPI_KEY\")\n        if not key:\n            sys.exit(\"No CDS key: set the CDSAPI_KEY environment variable (Colab: the CDSAPI_KEY secret).\")\n        started, done, failed = time.time(), 0, 0\n        with ThreadPoolExecutor(args.parallel) as pool:\n            futures = {pool.submit(download, j, weather, key): j for j in todo}\n            for f in as_completed(futures):\n                j = futures[f]\n                try:\n                    f.result()\n                    done += 1\n                except Exception as e:  # noqa: BLE001  one failed request must not stop the others\n                    failed += 1\n                    with open(weather / \"failed.txt\", \"a\") as log:\n                        log.write(f\"{time.strftime('%Y-%m-%d %H:%M')} {j['dataset']} {j['name']}: {str(e)[:300]}\\n\")\n                    print(f\"    {j['dataset']} {j['name']} failed: {str(e)[:200]}\", flush=True)\n                size = raw_path(weather, j).stat().st_size / 1e6 if raw_path(weather, j).exists() else 0\n                print(f\"... {done + failed}/{len(todo)}  {j['dataset']} {j['name']} ({size:.1f} MB)\"\n                      f\"{' FAILED' if not raw_path(weather, j).exists() else ''}, \"\n                      f\"{(time.time() - started) / 60:.0f} min\", flush=True)\n        if failed:\n            print(f\"{failed} requests failed (see {weather / 'failed.txt'}); run again to retry them.\")\n\n    print(\"\\nConverting\", flush=True)\n    wanted = set(pd.read_parquet(args.db / \"site_chips.parquet\")[\"chip_id\"])\n    convert(weather, wanted)\n\n\nif __name__ == \"__main__\":\n    main()\n",
    "scripts/era5_arco.py": "#!/usr/bin/env python3\n\"\"\"ERA5 single levels for every site, read from the ARCO-ERA5 copy on Google Cloud Storage.\n\nThe same 21 variables that era5.py asks the CDS for (see era5.DATASETS[\"single\"]), without the CDS\nqueue. ARCO-ERA5 (Google Research) stores the whole ERA5 archive as Zarr, one global field per\nvariable and hour, open to anyone. Reading one field costs the same whatever part of the world is\nkept, so the work is set by the hours needed, not by the number of sites:\n\n    for each site, its competition days and the day before each, from SOLAR_FROM to SOLAR_TO local\n    solar time (thermal hours with a margin); hours needed by several sites are read once\n\nEach month is written to WEATHER/arco/single/<year>-<month>.parquet in the tidy form of\nera5_single.parquet (chip_id, time, converted columns); era5.py merges these files into\nera5_single.parquet when it converts. A second run reads only the hours a month file is missing\n(new sites or days are added, nothing is read twice). Hours after the last date in ARCO are left\nfor the CDS (era5.py).\n\n    python3 scripts/era5_arco.py /content/drive/MyDrive/IGCS/database --threads 32\n\"\"\"\n\nimport argparse\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom era5 import CONVERT, DATASETS, site_days  # noqa: E402\n\nARCO_URL = \"gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3\"\nSOLAR_FROM, SOLAR_TO = 5, 21\n\n# ARCO variable (CDS name) -> netCDF short name used in CONVERT\nSHORT = {\n    \"boundary_layer_height\": \"blh\", \"cloud_base_height\": \"cbh\", \"convective_available_potential_energy\": \"cape\",\n    \"convective_inhibition\": \"cin\", \"k_index\": \"kx\", \"surface_sensible_heat_flux\": \"sshf\",\n    \"surface_latent_heat_flux\": \"slhf\", \"surface_solar_radiation_downwards\": \"ssrd\", \"total_cloud_cover\": \"tcc\",\n    \"low_cloud_cover\": \"lcc\", \"medium_cloud_cover\": \"mcc\", \"high_cloud_cover\": \"hcc\", \"2m_temperature\": \"t2m\",\n    \"2m_dewpoint_temperature\": \"d2m\", \"10m_u_component_of_wind\": \"u10\", \"10m_v_component_of_wind\": \"v10\",\n    \"100m_u_component_of_wind\": \"u100\", \"100m_v_component_of_wind\": \"v100\",\n    \"instantaneous_10m_wind_gust\": \"i10fg\", \"surface_pressure\": \"sp\", \"total_precipitation\": \"tp\",\n}\nassert set(SHORT) == set(DATASETS[\"single\"][\"variables\"])\n\n\ndef convert_values(name, x):\n    \"\"\"(column, values) for an ARCO variable, in the units of era5_single.parquet.\"\"\"\n    col, conv = CONVERT[SHORT[name]]\n    x = x.astype(\"float64\")\n    if conv == \"K\":\n        x = x - 273.15\n    elif isinstance(conv, float):\n        x = x * conv\n    return col, x.astype(\"float32\")\n\n\n# \u2500\u2500 What to read \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef needs(db, solar_from=SOLAR_FROM, solar_to=SOLAR_TO):\n    \"\"\"{UTC hour: array of chip_ids} for every site's days, local solar hours solar_from..solar_to.\"\"\"\n    chips = pd.read_parquet(db / \"site_chips.parquet\", columns=[\"site_id\", \"chip_id\", \"era5_lon\"])\n    site_chips = {s: g[\"chip_id\"].to_numpy() for s, g in chips.groupby(\"site_id\")}\n    site_lon = chips.groupby(\"site_id\")[\"era5_lon\"].mean()\n    hours = {}\n    for site, days in site_days(db).items():\n        if site not in site_chips:\n            continue\n        shift = site_lon[site] / 15.0  # solar time = UTC + longitude / 15\n        for d in days:\n            start = (d + pd.Timedelta(hours=solar_from - shift)).floor(\"h\")\n            end = (d + pd.Timedelta(hours=solar_to - shift)).ceil(\"h\")\n            for t in pd.date_range(start, end, freq=\"h\"):\n                hours.setdefault(t, []).append(site_chips[site])\n    return {t: np.unique(np.concatenate(v)) for t, v in sorted(hours.items())}\n\n\ndef arco_rc(chip_ids):\n    \"\"\"Row and column of each chip's ERA5 point in an ARCO field (latitude 90 \u2192 -90, longitude 0 \u2192 359.75).\"\"\"\n    from pta import grid\n\n    lat, lon = grid.chip_centre(np.asarray(chip_ids))\n    row = np.rint((90.0 - lat) / 0.25).astype(int)\n    col = np.rint((lon % 360.0) / 0.25).astype(int) % 1440\n    return row, col\n\n\n# \u2500\u2500 Reading \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n\ndef open_arco():\n    import xarray as xr\n\n    return xr.open_zarr(ARCO_URL, chunks=None, storage_options={\"token\": \"anon\"})\n\n\ndef last_hour(ds):\n    \"\"\"The last hour with data (the time axis runs to 2050; later hours are empty).\"\"\"\n    for key in (\"valid_time_stop_era5t\", \"valid_time_stop\"):\n        if key in ds.attrs:\n            try:\n                return pd.Timestamp(ds.attrs[key]) + pd.Timedelta(hours=23)\n            except (ValueError, TypeError):\n                pass\n    return None\n\n\ndef read_hours(ds, wanted, threads):\n    \"\"\"Tidy rows for {hour: chip_ids}: one global field per variable and hour, read `threads` at a time.\"\"\"\n    t0 = pd.Timestamp(ds.time.values[0])\n    jobs = [(t, name) for t in wanted for name in SHORT]\n\n    def one(job):\n        t, name = job\n        i = int((t - t0) / pd.Timedelta(hours=1))\n        field = ds[name].isel(time=i).values\n        row, col = arco_rc(wanted[t])\n        return t, name, field[row, col]\n\n    with ThreadPoolExecutor(threads) as pool:\n        got = list(pool.map(one, jobs))\n    frames = {}\n    for t, name, values in got:\n        f = frames.setdefault(t, {\"chip_id\": wanted[t], \"time\": np.full(len(wanted[t]), t.to_datetime64())})\n        col, x = convert_values(name, values)\n        f[col] = x\n    if not frames:\n        return pd.DataFrame()\n    df = pd.concat([pd.DataFrame(f) for f in frames.values()], ignore_index=True)\n    cols = [c for c in df.columns if c not in (\"chip_id\", \"time\")]\n    return df.dropna(subset=cols, how=\"all\")  # an hour ARCO does not have yet\n\n\ndef missing(path, wanted):\n    \"\"\"The hours of `wanted` the month file lacks (or lacks some chips of).\"\"\"\n    if not path.exists():\n        return dict(wanted)\n    have = pd.read_parquet(path, columns=[\"chip_id\", \"time\"])\n    have = have.groupby(\"time\")[\"chip_id\"].apply(set).to_dict()\n    return {t: c for t, c in wanted.items() if not set(c.tolist()) <= have.get(t, set())}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument(\"db\", type=Path, help=\"database folder (with sites from build_sites.py)\")\n    parser.add_argument(\"--weather\", type=Path, help=\"output folder (default DB/weather)\")\n    parser.add_argument(\"--threads\", type=int, default=32, help=\"fields read at a time (default 32)\")\n    parser.add_argument(\"--solar\", default=f\"{SOLAR_FROM}-{SOLAR_TO}\",\n                        help=f\"local solar hours to keep (default {SOLAR_FROM}-{SOLAR_TO})\")\n    parser.add_argument(\"--plan-only\", action=\"store_true\", help=\"print how much there is to read and stop\")\n    args = parser.parse_args()\n    out = (args.weather or args.db / \"weather\") / \"arco\" / \"single\"\n    out.mkdir(parents=True, exist_ok=True)\n    lo, hi = (int(x) for x in args.solar.split(\"-\"))\n\n    wanted = needs(args.db, lo, hi)\n    ds = open_arco()\n    absent = [n for n in SHORT if n not in ds]\n    if absent:\n        sys.exit(f\"ARCO-ERA5 does not have: {', '.join(absent)}\")\n    stop = last_hour(ds)\n    late = {t for t in wanted if stop is not None and t > stop}\n    by_month = {}\n    for t, c in wanted.items():\n        if t not in late:\n            by_month.setdefault((t.year, t.month), {})[t] = c\n    todo = {m: missing(out / f\"{m[0]}-{m[1]:02d}.parquet\", h) for m, h in by_month.items()}\n    todo = {m: h for m, h in todo.items() if h}\n    n_hours = sum(len(h) for h in todo.values())\n    print(f\"ARCO-ERA5: {len(wanted):,} UTC hours needed ({len({t.normalize() for t in wanted}):,} days, \"\n          f\"{len(by_month):,} months); {n_hours:,} hours in {len(todo):,} months to read \"\n          f\"= {n_hours * len(SHORT):,} global fields\", flush=True)\n    if late:\n        print(f\"  {len(late):,} hours are after ARCO's last date ({stop:%Y-%m-%d}); era5.py gets those from the CDS\",\n              flush=True)\n    if args.plan_only or not todo:\n        return\n\n    started, done = time.time(), 0\n    for (year, month), hours in sorted(todo.items()):\n        t = time.time()\n        df = read_hours(ds, hours, args.threads)\n        path = out / f\"{year}-{month:02d}.parquet\"\n        if path.exists():\n            df = pd.concat([pd.read_parquet(path), df], ignore_index=True)\n            df = df.drop_duplicates([\"chip_id\", \"time\"], keep=\"last\")\n        df = df.sort_values([\"chip_id\", \"time\"]).reset_index(drop=True)\n        tmp = path.with_suffix(\".part\")\n        df.to_parquet(tmp, index=False)\n        tmp.replace(path)\n        done += len(hours)\n        took = time.time() - t\n        rate = done / (time.time() - started)\n        print(f\"... {year}-{month:02d}: {len(hours)} hours, {df['chip_id'].nunique()} chips, \"\n              f\"{took:.0f} s ({len(hours) * len(SHORT) / max(took, 1e-9):.1f} fields/s) | \"\n              f\"{done:,}/{n_hours:,} hours, about {(n_hours - done) / rate / 3600:.1f} h left\", flush=True)\n    print(f\"Done in {(time.time() - started) / 3600:.1f} h. Run era5.py (or notebook 07) to merge them into \"\n          f\"era5_single.parquet.\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n",
}
for name, text in CODE.items():
    path = os.path.join(REPO, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)
print("Code written to", REPO)

## 3. What there is to read

How many hours and global fields, and whether some hours are newer than ARCO (those come from the CDS). Nothing is read yet.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--plan-only", "--solar", SOLAR)

## 4. Read

One line per month, with its speed and the time left. Stopping this cell also stops the reading; months already saved are kept.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--threads", str(THREADS), "--solar", SOLAR)

## 5. Build the tables

Merges these months (and any single-level files already downloaded from the CDS) into `weather/era5_single.parquet`; also converts whatever pressure-level and ERA5-Land files notebook 07 has downloaded so far.

In [ ]:
run(f"{REPO}/scripts/era5.py", DB, "--convert-only")

## 6. First check: ERA5 boundary layer vs real thermal tops

In [ ]:
import duckdb
db = duckdb.connect()
W = os.path.join(DB, "weather")
db.execute(f"CREATE VIEW era5_single AS SELECT * FROM read_parquet('{W}/era5_single.parquet')")
db.execute(f"CREATE VIEW climbs AS SELECT * FROM read_parquet('{DB}/climbs.parquet')")
db.execute(f"CREATE VIEW tasks AS SELECT * FROM read_parquet('{DB}/tasks.parquet')")

# First check against the pilots: ERA5 boundary-layer height vs the tops of real thermals, same chip and hour
import sys; sys.path.insert(0, REPO)
from pta import grid
c = db.sql("""
    SELECT c.task_key, c.lat, c.lon, c.alt_top, c.t_start, k.date
    FROM climbs c JOIN tasks k USING (task_key)
    WHERE c.is_thermal AND k.date IS NOT NULL
    USING SAMPLE 200000
""").df()
c["chip_id"] = grid.chip_id(c["lat"].to_numpy(), c["lon"].to_numpy())
c["time"] = (pd.to_datetime(c["date"]) + pd.to_timedelta(c["t_start"], unit="s")).dt.round("h")
w = db.sql("SELECT chip_id, time, blh_m FROM era5_single").df()
m = c.merge(w, on=["chip_id", "time"])
# thermal top above the ground (the chip's mean elevation from cells)
cells = pd.read_parquet(f"{DB}/cells.parquet", columns=["chip_id", "elev_mean"]).groupby("chip_id").elev_mean.mean()
m["ground"] = m["chip_id"].map(cells)
m["top_agl"] = m["alt_top"] - m["ground"]
print(f"{len(m):,} thermals matched to ERA5")
print(m[["top_agl", "blh_m"]].describe().round(0))
print("correlation:", round(m["top_agl"].corr(m["blh_m"]), 2))